In [7]:
import sys
import pandas as pd
import os
import argparse
import numpy as np

# 获取当前脚本所在目录的绝对路径（即子目录 scripts 的路径）
current_dir = os.path.dirname(os.path.abspath('/data/huangjh/code/Projects_PINN_LiB/PIKAN4SOH/Notebooks/correlation_heatmap.ipynb'))
# 获取上级目录路径（即 project_root/utils 的父目录 project_root）
parent_dir = os.path.dirname(current_dir)
# 将上级目录添加到模块搜索路径
sys.path.append(parent_dir)

from dataloader.dataloader import DF,XJTUdata,HUSTdata,MITdata,TJUdata
import data

# def get_args():
#     """设置参数"""
#     parser = argparse.ArgumentParser(description='计算数据集特征与SOH的皮尔逊相关系数')
#     parser.add_argument('--dataset',type=str,default='XJTU',help=['XJTU','TJU','MIT','HUST'])
#     parser.add_argument('--data_root', type=str, default='data/XJTU data', help='XJTU数据集根路径')
#     parser.add_argument('--batch_name', type=str, default='2C', help='目标批次名称（第一批次为"2C"）')
#     parser.add_argument('--normalization_method', type=str, default='min-max', help='归一化方法：min-max或z-score')
#     parser.add_argument('--batch_size',type=int,default=512)
#     args, _ = parser.parse_known_args()
#     return args

def calculate_pearson_correlation_xjtu(args):
    # 初始化XJTU数据加载器
    xjtu_data = XJTUdata(root=args.data_root, args=args)
    
    # 获取第一批次（batch_name）的所有CSV文件路径
    batch_files = [
        os.path.join(args.data_root, file) 
        for file in os.listdir(args.data_root) 
        if args.batch_name in file
    ]
    
    # 读取并合并所有批次数据
    dfs = []
    for file in batch_files:
        # 读取单个CSV文件（已处理异常值和归一化）
        df = xjtu_data.read_one_csv(file, nominal_capacity=2.0)  # 标称容量为2.0（XJTU数据集设定）
        dfs.append(df)
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # 确定特征列和SOH列（SOH为归一化后的容量）
    # 假设特征列是除"cycle index"和"capacity"外的所有列
    feature_columns = [col for col in combined_df.columns if col not in ['cycle index', 'capacity']]
    assert len(feature_columns) == 16, f"预期16个特征，实际找到{len(feature_columns)}个"
    
    # 计算每个特征与SOH的皮尔逊相关系数
    corr_results = {}
    for feature in feature_columns:
        corr = combined_df[feature].corr(combined_df['capacity'])
        corr_results[feature] = corr
    
    return corr_results

In [8]:
from contextlib import contextmanager

@contextmanager
def silent_output():
    """上下文管理器：临时抑制所有标准输出"""
    original_stdout = sys.stdout
    try:
        # 重定向输出到空设备
        sys.stdout = open(os.devnull, 'w')
        yield
    finally:
        # 恢复原始输出
        sys.stdout.close()
        sys.stdout = original_stdout

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [9]:
# 输出XJTU数据集中16个健康特征与SOH的皮尔逊相关系数
parser = argparse.ArgumentParser(description='计算数据集特征与SOH的皮尔逊相关系数')
parser.add_argument('--dataset',type=str,default='XJTU',help=['XJTU','TJU','MIT','HUST'])
parser.add_argument('--data_root', type=str, default='data/XJTU data', help='XJTU数据集根路径')
parser.add_argument('--batch_name', type=str, default='2C', help='目标批次名称（第一批次为"2C"）')
parser.add_argument('--normalization_method', type=str, default='min-max', help='归一化方法：min-max或z-score')
parser.add_argument('--batch_size',type=int,default=512)
args, _ = parser.parse_known_args()
args.dataset = 'XJTU'
args.data_root = '/data/huangjh/code/Projects_PINN_LiB/PIKAN4SOH/data/XJTU data'
corr_matrix = np.zeros(shape=(6,16),dtype=float)
for i,args.batch_name in enumerate(['2C', '3C', 'R2.5', 'R3', 'RW', 'satellite']):
    with silent_output():
        correlations = calculate_pearson_correlation_xjtu(args)
    print(f"XJTU数据集批次 {args.batch_name} 特征与SOH的皮尔逊相关系数：")
    for j,(feature, corr) in enumerate(correlations.items()):
        print(f"{feature}: {corr:.4f}")
        corr_matrix[i][j] = corr
    print('\n')
print(corr_matrix)
with open('../Data/xjtu_corr_matrix.npy', 'wb') as f:
    np.save(f, corr_matrix)

XJTU数据集批次 2C 特征与SOH的皮尔逊相关系数：
voltage mean: 0.6667
voltage std: -0.7206
voltage kurtosis: 0.8265
voltage skewness: -0.6045
CC Q: -0.3954
CC charge time: -0.3936
voltage slope: 0.2050
voltage entropy: -0.5319
current mean: 0.8379
current std: 0.7362
current kurtosis: -0.8337
current skewness: -0.8427
CV Q: -0.9662
CV charge time: -0.9726
current slope: -0.9182
current entropy: -0.9568


XJTU数据集批次 3C 特征与SOH的皮尔逊相关系数：
voltage mean: 0.8587
voltage std: -0.1525
voltage kurtosis: -0.1679
voltage skewness: -0.8671
CC Q: -0.7076
CC charge time: -0.7085
voltage slope: 0.2413
voltage entropy: -0.5927
current mean: 0.4718
current std: 0.4584
current kurtosis: -0.4968
current skewness: -0.4803
CV Q: -0.8431
CV charge time: -0.8648
current slope: -0.7916
current entropy: -0.8466


XJTU数据集批次 R2.5 特征与SOH的皮尔逊相关系数：
voltage mean: -0.2000
voltage std: -0.7534
voltage kurtosis: 0.8015
voltage skewness: 0.2611
CC Q: -0.2530
CC charge time: -0.2530
voltage slope: 0.2756
voltage entropy: -0.2440
current mean: 

In [10]:
def load_tju_batch(args):
    """加载TJU指定批次的所有CSV文件并合并数据"""
    # 获取第一批次的路径（假设批次按文件夹存储，如batch0, batch1等）
    batchs = ['Dataset_1_NCA_battery','Dataset_2_NCM_battery','Dataset_3_NCM_NCA_battery']
    target_batch = batchs[args.batch_idx]  # 第一批次索引为0
    batch_path = os.path.join(args.data_root, target_batch)
    
    # 获取批次内所有CSV文件路径
    csv_files = [os.path.join(batch_path, f) for f in os.listdir(batch_path) if f.endswith('.csv')]
    
    # 使用项目已有数据加载类读取并合并数据
    data_loader = DF(args=args)  # 初始化数据加载器（继承自项目中的基础类）
    dfs = []
    for file in csv_files:
        df = data_loader.read_one_csv(file, nominal_capacity=args.nominal_capacities[args.batch_idx])  # 读取并预处理单个文件
        dfs.append(df)
    
    # 合并所有电池数据（按循环索引拼接）
    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df

def calculate_pearson_correlation_tju(args):
    # 加载合并后的批次数据
    combined_df = load_tju_batch(args)
    
    # 确定特征列和SOH列（SOH为归一化后的容量）
    # 假设特征列为除"cycle index"和"capacity"外的所有列（根据实际数据集调整）
    feature_columns = [col for col in combined_df.columns if col not in ['cycle index', 'capacity']]
    assert len(feature_columns) == 16, f"预期16个特征，实际找到{len(feature_columns)}个"
    
    # 计算每个特征与SOH的皮尔逊相关系数
    corr_results = {}
    for feature in feature_columns:
        corr = combined_df[feature].corr(combined_df['capacity'])
        corr_results[feature] = corr
    
    return corr_results

In [11]:
# 输出TJU数据集中16个健康特征与SOH的皮尔逊相关系数
parser = argparse.ArgumentParser(description='计算数据集特征与SOH的皮尔逊相关系数')
parser.add_argument('--dataset',type=str,default='TJU',help=['XJTU','TJU','MIT','HUST'])
parser.add_argument('--data_root', type=str, default='data/TJU data', help='TJU数据集根路径')
parser.add_argument('--batch_idx', type=int, default=0, help=[0,1,2])
parser.add_argument('--nominal_capacities', type=list, default=[3.5,3.5,2.5], help='标称容量')
parser.add_argument('--normalization_method', type=str, default='min-max', help='归一化方法：min-max或z-score')
parser.add_argument('--batch_size',type=int,default=512)
args, _ = parser.parse_known_args()
args.dataset = 'TJU'
args.data_root = '/data/huangjh/code/Projects_PINN_LiB/PIKAN4SOH/data/TJU data'
corr_matrix = np.zeros(shape=(3,16),dtype=float)
for i,args.batch_idx in enumerate([0, 1, 2]):
    with silent_output():
        correlations = calculate_pearson_correlation_tju(args)
    print(f"TJU数据集批次 {args.batch_idx} 特征与SOH的皮尔逊相关系数：")
    for j,(feature, corr) in enumerate(correlations.items()):
        print(f"{feature}: {corr:.4f}") 
        corr_matrix[i][j] = corr
    print('\n')
print(corr_matrix)
with open('../Data/tju_corr_matrix.npy', 'wb') as f:
    np.save(f, corr_matrix)

TJU数据集批次 0 特征与SOH的皮尔逊相关系数：
voltage mean: 0.4316
voltage std: 0.3191
voltage kurtosis: -0.4556
voltage skewness: -0.4778
CC Q: 0.8222
CC charge time: 0.8222
voltage slope: -0.6432
voltage entropy: 0.6406
current mean: 0.7253
current std: 0.4080
current kurtosis: -0.7296
current skewness: -0.7538
CV Q: -0.7859
CV charge time: -0.7919
current slope: -0.7933
current entropy: -0.7860


TJU数据集批次 1 特征与SOH的皮尔逊相关系数：
voltage mean: -0.2299
voltage std: 0.3224
voltage kurtosis: -0.6741
voltage skewness: 0.6098
CC Q: 0.7812
CC charge time: 0.7809
voltage slope: -0.8306
voltage entropy: 0.5307
current mean: 0.8172
current std: 0.6176
current kurtosis: -0.7120
current skewness: -0.8161
CV Q: -0.7788
CV charge time: -0.7978
current slope: -0.8843
current entropy: -0.7539


TJU数据集批次 2 特征与SOH的皮尔逊相关系数：
voltage mean: 0.7345
voltage std: 0.5265
voltage kurtosis: -0.4703
voltage skewness: -0.7639
CC Q: 0.8707
CC charge time: 0.8703
voltage slope: -0.9008
voltage entropy: 0.8516
current mean: 0.8963
current 

In [12]:
def load_mit_all_data(args):
    """
    加载MIT数据集所有CSV文件并合并为全局DataFrame
    """
    data_loader = DF(args=args)
    all_dfs = []
    root = args.data_root
    for batch in ['2017-05-12', '2017-06-30', '2018-04-12']:
        batch_root = os.path.join(root, batch)
        files = os.listdir(batch_root)
        for f in files:
            file_path = os.path.join(batch_root, f)
            df = data_loader.read_one_csv(file_path, nominal_capacity=1.1)
            all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)
    return combined_df


def calculate_pearson_correlation_mit(combined_df):
    """
    计算16个特征与SOH的皮尔逊相关系数
    """
    feature_columns = [col for col in combined_df.columns if col not in ['cycle index', 'capacity']]
    assert len(feature_columns) == 16, f"预期16个特征，实际找到{len(feature_columns)}个"

    corr_results = {}
    for feature in feature_columns:
        corr = combined_df[feature].corr(combined_df['capacity'])
        corr_results[feature] = corr

    return corr_results


In [13]:
# 输出MIT数据集中16个健康特征与SOH的皮尔逊相关系数
parser = argparse.ArgumentParser(description='计算数据集特征与SOH的皮尔逊相关系数')
parser.add_argument('--dataset',type=str,default='MIT',help=['XJTU','TJU','MIT','HUST'])
parser.add_argument('--data_root', type=str, default='data/MIT data', help='MIT数据集根路径')
parser.add_argument('--normalization_method', type=str, default='min-max', help='归一化方法：min-max或z-score')
parser.add_argument('--batch_size',type=int,default=512)
args, _ = parser.parse_known_args()
args.dataset = 'MIT'
args.data_root = '/data/huangjh/code/Projects_PINN_LiB/PIKAN4SOH/data/MIT data'
combined_df = load_mit_all_data(args)
corr_matrix = np.zeros(shape=(16,),dtype=float)
with silent_output():
    correlations = calculate_pearson_correlation_mit(combined_df)
print(f"MIT数据集特征与SOH的皮尔逊相关系数：")
for j,(feature, corr) in enumerate(correlations.items()):
    print(f"{feature}: {corr:.4f}") 
    corr_matrix[j] = corr
print('\n')
print(corr_matrix)
with open('../Data/mit_corr_matrix.npy', 'wb') as f:
    np.save(f, corr_matrix)

MIT数据集特征与SOH的皮尔逊相关系数：
voltage mean: -0.8104
voltage std: -0.0476
voltage kurtosis: 0.6894
voltage skewness: 0.7487
CC Q: 0.9199
CC charge time: 0.9208
voltage slope: 0.5306
voltage entropy: 0.8390
current mean: -0.5568
current std: -0.3943
current kurtosis: 0.5079
current skewness: 0.5464
CV Q: 0.0518
CV charge time: 0.2375
current slope: -0.3363
current entropy: 0.1961


[-0.81043762 -0.04756341  0.68938769  0.74867702  0.9199377   0.92082966
  0.53057852  0.83902382 -0.55676704 -0.39427267  0.50792827  0.54644155
  0.05183652  0.23753918 -0.33633986  0.19607876]


In [14]:
def load_hust_all_data(args):
    """
    加载HUST数据集所有CSV文件并合并为全局DataFrame
    """
    # 初始化数据加载器（复用代码库中的DF类）
    data_loader = DF(args)  # 临时创建args对象
    data_loader.normalization_method = args.normalization_method  # 设置归一化方法

    # 获取所有HUST的CSV文件路径
    csv_files = [os.path.join(args.data_root, f) for f in os.listdir(args.data_root) if f.endswith('.csv')]

    # 读取并合并所有电池数据
    all_dfs = []
    for file in csv_files:
        # 读取单个CSV文件（已自动处理异常值和归一化）
        df = data_loader.read_one_csv(file, nominal_capacity=1.1)  
        all_dfs.append(df)

    # 合并所有电池的循环数据（按行拼接）
    combined_df = pd.concat(all_dfs, ignore_index=True)
    return combined_df

def calculate_pearson_correlation_hust(combined_df):
    """
    计算16个特征与SOH的皮尔逊相关系数
    """
    # 确定特征列和SOH列（假设SOH为'capacity'列）
    # 特征列：排除'cycle index'和'capacity'后的所有列（根据HUST实际列名调整）
    feature_columns = [col for col in combined_df.columns if col not in ['cycle index', 'capacity']]
    
    # 验证特征数量（确保为16个）
    assert len(feature_columns) == 16, f"预期16个特征，实际找到{len(feature_columns)}个"

    # 计算每个特征与SOH的皮尔逊相关系数
    corr_results = {}
    for feature in feature_columns:
        corr = combined_df[feature].corr(combined_df['capacity'])
        corr_results[feature] = corr

    return corr_results

In [15]:
# 输出HUST数据集中16个健康特征与SOH的皮尔逊相关系数
parser = argparse.ArgumentParser(description='计算数据集特征与SOH的皮尔逊相关系数')
parser.add_argument('--dataset',type=str,default='HUST',help=['XJTU','TJU','MIT','HUST'])
parser.add_argument('--data_root', type=str, default='data/HUST data', help='HUST数据集根路径')
parser.add_argument('--normalization_method', type=str, default='min-max', help='归一化方法：min-max或z-score')
parser.add_argument('--batch_size',type=int,default=512)
args, _ = parser.parse_known_args()
args.dataset = 'HUST'
args.data_root = '/data/huangjh/code/Projects_PINN_LiB/PIKAN4SOH/data/HUST data'
combined_df = load_hust_all_data(args)
corr_matrix = np.zeros(shape=(16,),dtype=float)
with silent_output():
    correlations = calculate_pearson_correlation_hust(combined_df)
print(f"HUST数据集特征与SOH的皮尔逊相关系数：")
for j,(feature, corr) in enumerate(correlations.items()):
    print(f"{feature}: {corr:.4f}") 
    corr_matrix[j] = corr
print('\n')
print(corr_matrix)
with open('../Data/hust_corr_matrix.npy', 'wb') as f:
    np.save(f, corr_matrix)

HUST数据集特征与SOH的皮尔逊相关系数：
voltage mean: -0.7549
voltage std: -0.7426
voltage kurtosis: 0.7664
voltage skewness: 0.7741
CC Q: 0.8278
CC charge time: 0.8278
voltage slope: 0.1141
voltage entropy: 0.8149
current mean: -0.5700
current std: -0.3656
current kurtosis: 0.6865
current skewness: 0.6649
CV Q: 0.4528
CV charge time: 0.5012
current slope: 0.3328
current entropy: 0.4932


[-0.75489068 -0.74257192  0.76641722  0.77409268  0.82777125  0.82781618
  0.11412879  0.81488699 -0.5699903  -0.36562942  0.68653675  0.66491483
  0.45281467  0.50123092  0.33283915  0.49315293]
